# MAJ-Debate: Human Evaluation Analysis

This notebook creates the annotation template, ingests completed human judgments, computes agreement, and exports real human-eval results. Demo/mock annotations have been removed.


## 0. Imports & Configuration


In [ ]:
import os, json
from pathlib import Path
from datetime import datetime
import pandas as pd

EVAL_SPLIT = 'human_eval'
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'notebooks').exists()), cwd)
STAGE4_PATH = PROJECT_ROOT / 'outputs' / 'stage4' / EVAL_SPLIT / 'stage4_judgments.json'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'human_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_TEMPLATE = OUT_DIR / 'annotations_template.csv'
ANNOTATIONS_INPUT = OUT_DIR / 'annotations_completed.csv'
RESULTS_FILE = OUT_DIR / 'human_eval_results.json'
N_ANNOTATORS = 3
print(f'Stage 4 input : {STAGE4_PATH}')
print(f'Template out  : {ANNOTATIONS_TEMPLATE}')
print(f'Results out   : {RESULTS_FILE}')


## 1. Create Annotation Template


In [ ]:
if not STAGE4_PATH.exists():
    raise FileNotFoundError(f'Stage 4 human_eval output not found: {STAGE4_PATH}')
with open(STAGE4_PATH, 'r', encoding='utf-8') as f:
    stage4 = json.load(f)
rows = []
for topic in stage4['topics']:
    for ann in range(1, N_ANNOTATORS + 1):
        rows.append({'topic_id': topic['topic_id'], 'topic_text': topic['topic_text'], 'graph_winner': topic['graph_winner'], 'judge_explanation': topic['explanation'], 'annotator_id': f'A{ann}', 'correct_label': '', 'persuasiveness_rating': '', 'notes': ''})
template_df = pd.DataFrame(rows)
template_df.to_csv(ANNOTATIONS_TEMPLATE, index=False)
template_df.head()


## 2. Agreement and Significance Helpers


In [ ]:
def majority_vote(labels):
    clean = [x for x in labels if str(x).strip()]
    if not clean:
        return 'UNLABELED'
    counts = {}
    for label in clean:
        counts[label] = counts.get(label, 0) + 1
    top = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    if len(top) > 1 and top[0][1] == top[1][1]:
        return 'TIE'
    return top[0][0]

def krippendorff_alpha_nominal(matrix):
    units = [list(filter(lambda x: str(x).strip() != '', row)) for row in matrix]
    units = [u for u in units if len(u) >= 2]
    if not units:
        return None
    categories = sorted({label for u in units for label in u})
    if len(categories) <= 1:
        return 1.0
    Do_num = 0.0
    Do_den = 0.0
    counts_total = {c: 0 for c in categories}
    for u in units:
        n = len(u)
        Do_den += n - 1
        counts = {c: u.count(c) for c in categories}
        Do_num += sum(counts[c] * (n - counts[c]) for c in categories)
        for c in categories:
            counts_total[c] += counts[c]
    Do = Do_num / Do_den if Do_den else 0.0
    N = sum(counts_total.values())
    if N <= 1:
        return None
    De = sum(counts_total[c] * (N - counts_total[c]) for c in categories) / (N - 1)
    return 1.0 if De == 0 else 1.0 - (Do / De)

def paired_permutation_pvalue(xs, ys, n_iter=2000):
    diffs = [x - y for x, y in zip(xs, ys)]
    observed = abs(sum(diffs))
    if not diffs:
        return None
    extreme = 0
    for i in range(n_iter):
        signed = [d if (i + j) % 2 == 0 else -d for j, d in enumerate(diffs)]
        if abs(sum(signed)) >= observed:
            extreme += 1
    return (extreme + 1) / (n_iter + 1)


## 3. Ingest Completed Annotations


In [ ]:
if not ANNOTATIONS_INPUT.exists():
    raise FileNotFoundError(f'Real human annotations not found: {ANNOTATIONS_INPUT}. Fill annotations_template.csv and save it as annotations_completed.csv.')
ann_df = pd.read_csv(ANNOTATIONS_INPUT)
required_cols = {'topic_id', 'graph_winner', 'correct_label', 'persuasiveness_rating'}
missing = required_cols - set(ann_df.columns)
if missing:
    raise ValueError(f'Missing required annotation columns: {sorted(missing)}')
ann_df.head()


## 4. Compute Results and Export


In [ ]:
topic_results = {}
matrix = []
graph_matches = []
for topic_id, group in ann_df.groupby('topic_id'):
    labels = list(group['correct_label'])
    matrix.append(labels)
    majority = majority_vote(labels)
    avg_pers = float(pd.to_numeric(group['persuasiveness_rating'], errors='coerce').dropna().mean()) if not group.empty else None
    graph_winner = group['graph_winner'].iloc[0]
    graph_matches.append(int(majority == graph_winner))
    topic_results[topic_id] = {'majority_label': majority, 'graph_winner': graph_winner, 'agreement_with_graph': int(majority == graph_winner), 'avg_persuasiveness': avg_pers, 'n_annotations': int(len(group))}
alpha = krippendorff_alpha_nominal(matrix)
p_value = paired_permutation_pvalue(graph_matches, [0 for _ in graph_matches])
results_doc = {'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'), 'annotation_source': str(ANNOTATIONS_INPUT), 'n_topics': len(topic_results), 'n_annotators': N_ANNOTATORS, 'krippendorff_alpha': alpha, 'paired_permutation_pvalue_vs_zero_baseline': p_value, 'topic_results': topic_results}
with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
    json.dump(results_doc, f, indent=2)
print(f'Saved: {RESULTS_FILE}')
print(json.dumps({'n_topics': results_doc['n_topics'], 'krippendorff_alpha': results_doc['krippendorff_alpha'], 'paired_permutation_pvalue_vs_zero_baseline': results_doc['paired_permutation_pvalue_vs_zero_baseline']}, indent=2))
